# Individual Data Analysis & Cleaning — Thisara

**Member:** Thisara  
**Downstream Focus:** Crime Types, Offences & Victims (RQs 4–6)  
**Allowed Libraries Only:** `numpy`, `pandas`, `matplotlib`, `seaborn`, `geopandas`  
**Workflow:** Each member (Thisara, Sachithra, Kavindu) performs independent data exploration and cleaning in their own notebook. We then review all three notebooks together and merge the best techniques into one final master cleaning pipeline.

> **Reference:** Check `docs/Dataset_Domain_Brief.md`, `docs/Dataset_Domain_Knowledge_Sinhala.md`, and `notebooks/00_data_cleaning_thisara.ipynb` anytime for column definitions and domain traps.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

sys.path.append(str(Path("..").resolve()))
from utils.viz_style import apply_style, save_fig, RAW_DATA_PATH, REPO_ROOT

apply_style()
pd.set_option("display.max_columns", 50)
print("Raw dataset path:", RAW_DATA_PATH)


## 1. Load Raw Dataset & Initial Overview
Inspect shape, column names, data types, and sample rows.


In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(3)


## 2. Missing Values, Duplicates & NIBRS Granularity Audit
Quantify missing % per column, exact duplicates, and Incident ID vs. Offence row counts.


In [ ]:
# Missing value summary table
missing_df = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True),
    "dtype": df_raw.dtypes.astype(str),
}).sort_values("missing_pct", ascending=False)

display(missing_df)
print("Exact duplicates:", df_raw.duplicated().sum())
print("Unique Incident IDs:", df_raw["Incident ID"].nunique())


## 3. Temporal Analysis & Datetime Cleaning
Parse `Start_Date_Time`, `End_Date_Time`, and `Dispatch Date / Time`. Check partial years (2016/2026), negative durations (`End < Start`), `Dispatch < Start`, and `00:00:00` midnight spikes.


In [ ]:
df = df_raw.copy()
dt_cols = ["Dispatch Date / Time", "Start_Date_Time", "End_Date_Time"]
for col in dt_cols:
    df[col] = pd.to_datetime(df[col], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

# TODO: Add your temporal anomaly checks, visualisations, and cleaning rules here


## 4. Categorical & Crime Taxonomy Cleaning
Inspect `Crime Name1/2/3`, `NIBRS Code` (`90Z`), `Agency`, `Police District Name/Number`, `City`, `State`, and `Place`. Document your cleaning decisions.


In [ ]:
# TODO: Inspect and clean categorical columns (whitespace, casing, State numeric noise, Police District Name nulls, etc.)


## 5. Spatial Audit & Cleaning with GeoPandas
Inspect `(0.0, 0.0)` Null Island coordinates and build a `GeoDataFrame` (`EPSG:4326`) on valid Montgomery County points.


In [ ]:
# Identify (0, 0) sentinel coordinates
zero_coord_mask = (df["Latitude"] == 0) | (df["Longitude"] == 0)
print(f"(0, 0) Sentinel coords: {zero_coord_mask.sum():,} ({zero_coord_mask.mean()*100:.2f}%)")

# Example GeoPandas conversion on valid points (replace 0 with NaN first in df)
df.loc[zero_coord_mask, ["Latitude", "Longitude"]] = np.nan
valid_geo = df.dropna(subset=["Latitude", "Longitude"])
gdf = gpd.GeoDataFrame(
    valid_geo,
    geometry=gpd.points_from_xy(valid_geo["Longitude"], valid_geo["Latitude"]),
    crs="EPSG:4326",
)
print("Valid GeoDataFrame rows:", len(gdf))

# TODO: Add spatial checks / plots with GeoPandas + Matplotlib/Seaborn


## 6. Missing-Data Method Comparison & Wrangling
Compare at least 2 cleaning/imputation approaches (e.g., drop vs. flag/impute), quantify exact `% of records affected`, and engineer analytical features (`Year`, `Month`, `DayOfWeek`, `Hour`, etc.).


In [ ]:
# TODO: Compare cleaning methods and construct your candidate cleaned DataFrame (`df_clean`)
df_clean = df.copy()


## 7. Export Candidate Cleaned Dataset & Review Summary (Thisara)
Export as `.csv.gz` (native `pandas` compression, no extra libraries required) and print a summary table for the 3-member merge review.


In [ ]:
out_path = REPO_ROOT / "data" / "processed" / "cleaned_candidate_thisara.csv.gz"
out_path.parent.mkdir(parents=True, exist_ok=True)

# Uncomment when ready to export your candidate cleaned dataset:
# df_clean.to_csv(out_path, index=False, compression="gzip")
# print(f"Saved candidate dataset -> {out_path}")

print("Summary for Team Merge Review (Thisara):")
print(f"  Raw rows:     {len(df_raw):,}")
print(f"  Cleaned rows: {len(df_clean):,} ({(1 - len(df_clean)/len(df_raw))*100:.2f}% removed)")
print(f"  Columns:      {df_clean.shape[1]}")
